# Therapy DC-RS: Comparing Continual Learning vs Static Memory

This notebook compares **three conditions** for therapeutic conversation memory:

| Condition | Memory Type | Context Window | Description |
|-----------|-------------|----------------|-------------|
| **Baseline** | None | Last 10 turns | Sliding window context only |
| **Static mem0** | Raw memories | **None** | Relies purely on retrieved memories |
| **DC-RS** | Curated strategies | **None** | Relies purely on learned strategies |

## Paper Narrative

**Continual learning for long-context, multi-turn therapeutic conversations**: 
We demonstrate that continual learning during test-time (DC-RS) is more suitable than 
static memory (mem0) for maintaining therapeutic alignment over extended conversations.

## Key Hypothesis

DC-RS will outperform both baseline and mem0 because:
1. **Curated strategies** prevent distortion accumulation (unlike mem0)
2. **Test-time learning** adapts to patient patterns (unlike baseline)
3. **Clinical filtering** maintains professional boundaries

## Experimental Design

- **Baseline**: Only gets sliding window (last 10 turns) - no memory system
- **Mem0**: No context window - must rely entirely on retrieved memories
- **DC-RS**: No context window - must rely entirely on curated cheatsheet strategies

This tests whether memory systems can work **without** the sliding window crutch.

## 1. Setup and Imports

In [ ]:
import sys
import os
import json
import time
import shutil
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field, asdict
from datetime import datetime

# Add our-pipeline to path
pipeline_path = Path("./our-pipeline")
if str(pipeline_path) not in sys.path:
    sys.path.insert(0, str(pipeline_path))

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Import existing modules
from transcript_parser import (
    parse_html_transcript_text,
    get_counselor_turns,
    get_patient_turns,
    ConversationTurn,
    get_conversation_context
)
from therapeutic_framework import CBT_SYSTEM_PROMPT
from alignment_evaluators import (
    create_openai_client,
    create_ollama_client,
    evaluate_cbt_adherence,
    evaluate_persona_consistency,
    calculate_statistics,
    parse_json_response
)

# Import Dynamic Cheatsheet module
from dynamic_cheatsheet import (
    TherapeuticCheatsheet,
    ExtractionResult,
    DCRSResult,
    extract_strategies,
    generate_with_cheatsheet,
    generate_baseline,
    generate_with_memories,
    analyze_cheatsheet_evolution
)

# Import Mem0 for comparison
from mem0_integration import (
    initialize_mem0,
    create_mem0_config_with_llm,
    add_conversation_turn_to_memory,
    get_all_memories,
    format_memories_for_context
)

# Output directory
OUTPUT_DIR = Path("./output_therapy_dc_rs")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "images").mkdir(exist_ok=True)
(OUTPUT_DIR / "checkpoints").mkdir(exist_ok=True)
(OUTPUT_DIR / "results").mkdir(exist_ok=True)

print("All modules loaded successfully!")
print(f"Output directory: {OUTPUT_DIR}")

## 2. Model Configuration

In [ ]:
# ============================================================================
# MODEL CONFIGURATION
# ============================================================================

# OPTION A: Use Lambda Cloud GPU instance
USE_LAMBDA_CLOUD = True
LAMBDA_CLOUD_BASE_URL = "http://localhost:11434/v1"  # SSH tunnel
LAMBDA_CLOUD_MODEL = "gpt-oss:20b"

# OPTION B: Use Ollama locally
USE_OLLAMA = False
OLLAMA_MODEL = "llama3.1:8b"

# OPTION C: Use OpenAI API
USE_OPENAI = False
OPENAI_MODEL = "gpt-4o-mini"

# ============================================================================
# Create the client
# ============================================================================

if USE_LAMBDA_CLOUD:
    from openai import OpenAI
    client = OpenAI(
        base_url=LAMBDA_CLOUD_BASE_URL,
        api_key="lambda"
    )
    MODEL = LAMBDA_CLOUD_MODEL
    print(f"Using Lambda Cloud GPU instance")
    print(f"  Base URL: {LAMBDA_CLOUD_BASE_URL}")
    print(f"  Model: {MODEL}")
    print("Make sure SSH tunnel is active: ssh -L 11434:localhost:11434 ubuntu@<ip>")
elif USE_OLLAMA:
    client = create_ollama_client()
    MODEL = OLLAMA_MODEL
    print(f"Using Ollama with model: {MODEL}")
elif USE_OPENAI:
    client = create_openai_client()
    MODEL = OPENAI_MODEL
    print(f"Using OpenAI with model: {MODEL}")
else:
    raise ValueError("Please set one of USE_LAMBDA_CLOUD, USE_OLLAMA, or USE_OPENAI to True")

print("\nClient created successfully!")

## 3. Initialize Mem0 (for comparison condition)

In [ ]:
# ============================================================================
# MEM0 CONFIGURATION
# ============================================================================

RESET_MEM0 = True  # Set to True for fresh comparison
CHROMA_DB_PATH = "./chroma_db_therapy_dcrs_comparison"
CHROMA_COLLECTION_NAME = "therapy_dcrs_comparison"
USER_ID = "patient_dcrs_comparison"

# Delete existing ChromaDB if reset
if RESET_MEM0 and Path(CHROMA_DB_PATH).exists():
    shutil.rmtree(CHROMA_DB_PATH)
    print(f"Deleted existing {CHROMA_DB_PATH} for fresh start")

# Configure Mem0
if USE_LAMBDA_CLOUD:
    mem_config = create_mem0_config_with_llm(
        llm_provider="ollama",
        model=LAMBDA_CLOUD_MODEL,
        base_url=LAMBDA_CLOUD_BASE_URL.replace("/v1", "")
    )
elif USE_OLLAMA:
    mem_config = create_mem0_config_with_llm(
        llm_provider="ollama",
        model=OLLAMA_MODEL,
        base_url="http://localhost:11434"
    )
else:
    mem_config = create_mem0_config_with_llm(
        llm_provider="openai",
        model=OPENAI_MODEL
    )

mem_config["vector_store"]["config"]["collection_name"] = CHROMA_COLLECTION_NAME
mem_config["vector_store"]["config"]["path"] = CHROMA_DB_PATH

memory = initialize_mem0(config=mem_config, reset_collection=RESET_MEM0)
print(f"Mem0 initialized for comparison")
print(f"  Collection: {CHROMA_COLLECTION_NAME}")
print(f"  Path: {CHROMA_DB_PATH}")

## 4. Load Transcript Data

In [ ]:
import re

COMBINED_TRANSCRIPT_PATH = Path("./0518-014_combined_transcript.txt")

print(f"Loading combined transcript: {COMBINED_TRANSCRIPT_PATH}")

# Read and parse
with open(COMBINED_TRANSCRIPT_PATH, 'r', encoding='utf-8') as f:
    combined_content = f.read()

# Split by transcript sections
transcript_pattern = r'==========\s*(\d+\.txt)\s*=========='
sections = re.split(transcript_pattern, combined_content)

# Parse sections
transcript_sections = []
current_filename = None
for section in sections:
    if re.match(r'\d+\.txt', section.strip()):
        current_filename = section.strip()
    elif current_filename and section.strip():
        transcript_sections.append({
            'filename': current_filename,
            'content': section
        })
        current_filename = None

# Parse all turns
all_turns = []
turn_to_transcript_map = {}
transcript_boundaries = []

global_turn_number = 0
for ts in transcript_sections:
    try:
        section_turns = parse_html_transcript_text(ts['content'])
    except ValueError:
        continue
    
    start_turn = global_turn_number + 1
    
    for turn in section_turns:
        global_turn_number += 1
        turn.turn_number = global_turn_number
        turn_to_transcript_map[global_turn_number] = ts['filename']
        all_turns.append(turn)
    
    end_turn = global_turn_number
    if end_turn >= start_turn:
        transcript_boundaries.append({
            'filename': ts['filename'],
            'start_turn': start_turn,
            'end_turn': end_turn,
            'turn_count': end_turn - start_turn + 1
        })

print(f"\nTotal turns: {len(all_turns)}")
print(f"Counselor turns: {len(get_counselor_turns(all_turns))}")
print(f"Patient turns: {len(get_patient_turns(all_turns))}")
print(f"Sessions: {len(transcript_boundaries)}")

## 5. Experiment Configuration

In [ ]:
# ============================================================================
# EXPERIMENT CONFIGURATION
# ============================================================================

MAX_TURNS_TO_PROCESS = 30  # Start small for testing, increase for full experiment
DELAY_BETWEEN_CALLS = 0.2  # Seconds between API calls
VERBOSE = True

# Conditions to run
RUN_BASELINE = True
RUN_MEM0 = True
RUN_DCRS = True

print(f"Experiment Configuration:")
print(f"  Max turns to process: {MAX_TURNS_TO_PROCESS}")
print(f"  Model: {MODEL}")
print(f"  Delay between calls: {DELAY_BETWEEN_CALLS}s")
print(f"  Conditions: Baseline={RUN_BASELINE}, Mem0={RUN_MEM0}, DC-RS={RUN_DCRS}")

## 6. Run All Three Conditions

In [ ]:
def truncate(text, length=80):
    """Truncate text for display."""
    return text[:length] + "..." if len(text) > length else text


def run_condition(
    condition_name: str,
    client,
    turns: List[ConversationTurn],
    model: str,
    max_turns: int,
    delay: float,
    verbose: bool,
    memory=None,  # For mem0 condition
    user_id: str = None
) -> Dict[str, Any]:
    """
    Run a single experimental condition.
    
    Args:
        condition_name: "baseline", "mem0", or "dcrs"
        client: OpenAI client
        turns: All conversation turns
        model: Model name
        max_turns: Max patient turns to process
        delay: Delay between API calls
        verbose: Print progress
        memory: Mem0 memory object (for mem0 condition)
        user_id: User ID for mem0
    """
    print(f"\n{'='*60}")
    print(f"RUNNING CONDITION: {condition_name.upper()}")
    print(f"{'='*60}")
    
    patient_turns = get_patient_turns(turns)[:max_turns]
    baseline_response = "I hear that you're experiencing some difficulties. Can you tell me more?"
    
    # Initialize condition-specific state
    cheatsheet = TherapeuticCheatsheet() if condition_name == "dcrs" else None
    
    results = {
        "condition": condition_name,
        "generated_responses": [],
        "cbt_evaluations": [],
        "persona_evaluations": [],
        "cheatsheet_snapshots": [],
        "memory_snapshots": []
    }
    
    for i, patient_turn in enumerate(patient_turns):
        if verbose:
            print(f"\n[Turn {patient_turn.turn_number}] {condition_name}...")
        
        # Only baseline gets sliding window context
        # Mem0 and DC-RS rely purely on their memory systems
        context = get_conversation_context(turns, patient_turn.turn_number, max_turns=10)
        
        # ====================================================================
        # GENERATE RESPONSE (condition-specific)
        # ====================================================================
        if condition_name == "baseline":
            # Baseline: Uses sliding window context ONLY
            generated = generate_baseline(
                client=client,
                patient_turn=patient_turn.content,
                conversation_context=context,
                model=model
            )
        
        elif condition_name == "mem0":
            # Mem0: Uses retrieved memories ONLY (no sliding window)
            current_memories = get_all_memories(memory, user_id)
            memories_context = format_memories_for_context(current_memories)
            
            generated = generate_with_memories(
                client=client,
                patient_turn=patient_turn.content,
                conversation_context="",  # NO CONTEXT - memories only
                memories_context=memories_context,
                model=model
            )
            
            # Add turn to mem0
            add_conversation_turn_to_memory(
                memory=memory,
                turn_content=patient_turn.content,
                role="patient",
                turn_number=patient_turn.turn_number,
                user_id=user_id,
                verbose=False
            )
            add_conversation_turn_to_memory(
                memory=memory,
                turn_content=generated,
                role="counselor",
                turn_number=patient_turn.turn_number,
                user_id=user_id,
                verbose=False
            )
            
            # Snapshot memory state
            updated_memories = get_all_memories(memory, user_id)
            results["memory_snapshots"].append({
                "turn_number": patient_turn.turn_number,
                "memory_count": len(updated_memories)
            })
        
        elif condition_name == "dcrs":
            # DC-RS: Uses cheatsheet strategies ONLY (no sliding window)
            generated = generate_with_cheatsheet(
                client=client,
                patient_turn=patient_turn.content,
                conversation_context="",  # NO CONTEXT - cheatsheet only
                cheatsheet=cheatsheet,
                model=model
            )
            
            time.sleep(delay)
            
            # Extract strategies (TEST-TIME LEARNING)
            cheatsheet, extraction = extract_strategies(
                client=client,
                patient_turn=patient_turn.content,
                counselor_response=generated,
                current_cheatsheet=cheatsheet,
                turn_number=patient_turn.turn_number,
                model=model
            )
            
            # Snapshot cheatsheet state
            results["cheatsheet_snapshots"].append({
                "turn_number": patient_turn.turn_number,
                **cheatsheet.get_stats(),
                "extracted_this_turn": extraction.get_extracted_count(),
                "filtered_this_turn": len(extraction.filtered_content)
            })
            
            if verbose:
                print(f"    Cheatsheet: {cheatsheet.get_stats()['total']} strategies")
        
        results["generated_responses"].append({
            "turn_number": patient_turn.turn_number,
            "patient_turn": patient_turn.content[:200],
            "generated_response": generated
        })
        
        if verbose:
            print(f"    Generated: {truncate(generated, 70)}")
        
        time.sleep(delay)
        
        # ====================================================================
        # EVALUATE RESPONSE
        # ====================================================================
        cbt_result = evaluate_cbt_adherence(
            client=client,
            counselor_response=generated,
            conversation_context=context,
            turn_number=patient_turn.turn_number,
            model=model
        )
        results["cbt_evaluations"].append(asdict(cbt_result))
        
        time.sleep(delay)
        
        persona_result = evaluate_persona_consistency(
            client=client,
            counselor_response=generated,
            baseline_response=baseline_response,
            conversation_context=context,
            turn_number=patient_turn.turn_number,
            model=model
        )
        results["persona_evaluations"].append(asdict(persona_result))
        
        if verbose:
            print(f"    Scores: CBT={cbt_result.score}/10 | Persona={persona_result.score}/10")
        
        time.sleep(delay)
    
    # Calculate summary statistics
    cbt_scores = [r["score"] for r in results["cbt_evaluations"]]
    persona_scores = [r["score"] for r in results["persona_evaluations"]]
    
    results["summary"] = {
        "cbt_mean": sum(cbt_scores) / len(cbt_scores) if cbt_scores else 0,
        "cbt_min": min(cbt_scores) if cbt_scores else 0,
        "cbt_max": max(cbt_scores) if cbt_scores else 0,
        "persona_mean": sum(persona_scores) / len(persona_scores) if persona_scores else 0,
        "persona_min": min(persona_scores) if persona_scores else 0,
        "persona_max": max(persona_scores) if persona_scores else 0,
        "num_turns": len(cbt_scores)
    }
    
    # Add final cheatsheet for DC-RS
    if condition_name == "dcrs" and cheatsheet:
        results["final_cheatsheet"] = cheatsheet.to_dict()
    
    print(f"\n{condition_name.upper()} Complete!")
    print(f"  CBT Mean: {results['summary']['cbt_mean']:.2f}")
    print(f"  Persona Mean: {results['summary']['persona_mean']:.2f}")
    
    return results


print("Condition runner defined.")

In [ ]:
# Store all results
all_results = {}

# ============================================================================
# RUN BASELINE CONDITION
# ============================================================================
if RUN_BASELINE:
    all_results["baseline"] = run_condition(
        condition_name="baseline",
        client=client,
        turns=all_turns,
        model=MODEL,
        max_turns=MAX_TURNS_TO_PROCESS,
        delay=DELAY_BETWEEN_CALLS,
        verbose=VERBOSE
    )

In [ ]:
# ============================================================================
# RUN MEM0 CONDITION
# ============================================================================
if RUN_MEM0:
    all_results["mem0"] = run_condition(
        condition_name="mem0",
        client=client,
        turns=all_turns,
        model=MODEL,
        max_turns=MAX_TURNS_TO_PROCESS,
        delay=DELAY_BETWEEN_CALLS,
        verbose=VERBOSE,
        memory=memory,
        user_id=USER_ID
    )

In [ ]:
# ============================================================================
# RUN DC-RS CONDITION
# ============================================================================
if RUN_DCRS:
    all_results["dcrs"] = run_condition(
        condition_name="dcrs",
        client=client,
        turns=all_turns,
        model=MODEL,
        max_turns=MAX_TURNS_TO_PROCESS,
        delay=DELAY_BETWEEN_CALLS,
        verbose=VERBOSE
    )

## 7. Compare Results

In [ ]:
print("="*80)
print("COMPARISON: BASELINE vs MEM0 vs DC-RS")
print("="*80)

# Build comparison table
conditions = []
if "baseline" in all_results:
    conditions.append(("Baseline", all_results["baseline"]["summary"]))
if "mem0" in all_results:
    conditions.append(("Mem0", all_results["mem0"]["summary"]))
if "dcrs" in all_results:
    conditions.append(("DC-RS", all_results["dcrs"]["summary"]))

print(f"\n{'Metric':<25} ", end="")
for name, _ in conditions:
    print(f"{name:<15}", end="")
print()
print("-" * (25 + 15 * len(conditions)))

# CBT Adherence
print(f"{'CBT Adherence (Mean)':<25} ", end="")
for _, summary in conditions:
    print(f"{summary['cbt_mean']:<15.2f}", end="")
print()

print(f"{'CBT Adherence (Min)':<25} ", end="")
for _, summary in conditions:
    print(f"{summary['cbt_min']:<15}", end="")
print()

print(f"{'CBT Adherence (Max)':<25} ", end="")
for _, summary in conditions:
    print(f"{summary['cbt_max']:<15}", end="")
print()

print()

# Persona Consistency
print(f"{'Persona Consistency (Mean)':<25} ", end="")
for _, summary in conditions:
    print(f"{summary['persona_mean']:<15.2f}", end="")
print()

print(f"{'Persona Consistency (Min)':<25} ", end="")
for _, summary in conditions:
    print(f"{summary['persona_min']:<15}", end="")
print()

print(f"{'Persona Consistency (Max)':<25} ", end="")
for _, summary in conditions:
    print(f"{summary['persona_max']:<15}", end="")
print()

# Calculate improvements
if "baseline" in all_results and "dcrs" in all_results:
    baseline_cbt = all_results["baseline"]["summary"]["cbt_mean"]
    dcrs_cbt = all_results["dcrs"]["summary"]["cbt_mean"]
    baseline_persona = all_results["baseline"]["summary"]["persona_mean"]
    dcrs_persona = all_results["dcrs"]["summary"]["persona_mean"]
    
    print(f"\nDC-RS vs Baseline Improvement:")
    print(f"  CBT: {dcrs_cbt - baseline_cbt:+.2f}")
    print(f"  Persona: {dcrs_persona - baseline_persona:+.2f}")

if "mem0" in all_results and "dcrs" in all_results:
    mem0_cbt = all_results["mem0"]["summary"]["cbt_mean"]
    dcrs_cbt = all_results["dcrs"]["summary"]["cbt_mean"]
    mem0_persona = all_results["mem0"]["summary"]["persona_mean"]
    dcrs_persona = all_results["dcrs"]["summary"]["persona_mean"]
    
    print(f"\nDC-RS vs Mem0 Improvement:")
    print(f"  CBT: {dcrs_cbt - mem0_cbt:+.2f}")
    print(f"  Persona: {dcrs_persona - mem0_persona:+.2f}")

In [ ]:
# Display DC-RS final cheatsheet
if "dcrs" in all_results and "final_cheatsheet" in all_results["dcrs"]:
    print("="*60)
    print("DC-RS FINAL CHEATSHEET")
    print("="*60)
    
    fc = all_results["dcrs"]["final_cheatsheet"]
    
    if fc["cbt_techniques"]:
        print("\nCBT Techniques:")
        for t in fc["cbt_techniques"]:
            print(f"  - {t}")
    
    if fc["distortion_patterns"]:
        print("\nDistortion Patterns:")
        for p in fc["distortion_patterns"]:
            print(f"  - {p}")
    
    if fc["effective_interventions"]:
        print("\nEffective Interventions:")
        for i in fc["effective_interventions"]:
            print(f"  - {i}")
    
    print(f"\nTotal strategies: {fc['stats']['total']}")

## 8. Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = {"baseline": "red", "mem0": "orange", "dcrs": "blue"}
labels = {"baseline": "Baseline", "mem0": "Static Mem0", "dcrs": "DC-RS"}

# ============================================================================
# Plot 1: CBT Adherence Over Time
# ============================================================================
ax1 = axes[0, 0]
for condition_name, results in all_results.items():
    turns = [r["turn_number"] for r in results["cbt_evaluations"]]
    scores = [r["score"] for r in results["cbt_evaluations"]]
    ax1.plot(turns, scores, color=colors[condition_name], alpha=0.4, linewidth=1)
    
    # Rolling average
    window = min(5, len(scores) // 3) or 2
    if len(scores) >= window:
        rolling = np.convolve(scores, np.ones(window)/window, mode='valid')
        ax1.plot(turns[window//2:len(rolling)+window//2], rolling, 
                color=colors[condition_name], linewidth=2.5, label=labels[condition_name])

ax1.axhline(y=7, color='green', linestyle='--', alpha=0.5, label='Good Threshold')
ax1.set_xlabel('Turn Number')
ax1.set_ylabel('CBT Adherence Score')
ax1.set_title('CBT Adherence Over Time')
ax1.legend(loc='lower left')
ax1.set_ylim(0, 11)
ax1.grid(True, alpha=0.3)

# ============================================================================
# Plot 2: Persona Consistency Over Time
# ============================================================================
ax2 = axes[0, 1]
for condition_name, results in all_results.items():
    turns = [r["turn_number"] for r in results["persona_evaluations"]]
    scores = [r["score"] for r in results["persona_evaluations"]]
    ax2.plot(turns, scores, color=colors[condition_name], alpha=0.4, linewidth=1)
    
    window = min(5, len(scores) // 3) or 2
    if len(scores) >= window:
        rolling = np.convolve(scores, np.ones(window)/window, mode='valid')
        ax2.plot(turns[window//2:len(rolling)+window//2], rolling, 
                color=colors[condition_name], linewidth=2.5, label=labels[condition_name])

ax2.axhline(y=7, color='green', linestyle='--', alpha=0.5, label='Good Threshold')
ax2.set_xlabel('Turn Number')
ax2.set_ylabel('Persona Consistency Score')
ax2.set_title('Persona Consistency Over Time')
ax2.legend(loc='lower left')
ax2.set_ylim(0, 11)
ax2.grid(True, alpha=0.3)

# ============================================================================
# Plot 3: Memory/Strategy Growth
# ============================================================================
ax3 = axes[1, 0]

if "dcrs" in all_results and all_results["dcrs"]["cheatsheet_snapshots"]:
    dcrs_turns = [s["turn_number"] for s in all_results["dcrs"]["cheatsheet_snapshots"]]
    dcrs_total = [s["total"] for s in all_results["dcrs"]["cheatsheet_snapshots"]]
    ax3.plot(dcrs_turns, dcrs_total, 'b-', linewidth=2, label='DC-RS Strategies')
    ax3.fill_between(dcrs_turns, 0, dcrs_total, alpha=0.2, color='blue')

if "mem0" in all_results and all_results["mem0"]["memory_snapshots"]:
    mem0_turns = [s["turn_number"] for s in all_results["mem0"]["memory_snapshots"]]
    mem0_count = [s["memory_count"] for s in all_results["mem0"]["memory_snapshots"]]
    ax3.plot(mem0_turns, mem0_count, 'orange', linewidth=2, label='Mem0 Memories')
    ax3.fill_between(mem0_turns, 0, mem0_count, alpha=0.2, color='orange')

ax3.set_xlabel('Turn Number')
ax3.set_ylabel('Count')
ax3.set_title('Memory/Strategy Accumulation')
ax3.legend()
ax3.grid(True, alpha=0.3)

# ============================================================================
# Plot 4: Mean Scores Comparison (Bar Chart)
# ============================================================================
ax4 = axes[1, 1]

condition_names = list(all_results.keys())
x = np.arange(len(condition_names))
width = 0.35

cbt_means = [all_results[c]["summary"]["cbt_mean"] for c in condition_names]
persona_means = [all_results[c]["summary"]["persona_mean"] for c in condition_names]

bars1 = ax4.bar(x - width/2, cbt_means, width, label='CBT Adherence', 
               color=[colors[c] for c in condition_names], alpha=0.7)
bars2 = ax4.bar(x + width/2, persona_means, width, label='Persona Consistency',
               color=[colors[c] for c in condition_names], alpha=0.4, hatch='//')

ax4.axhline(y=7, color='green', linestyle='--', alpha=0.5, label='Good Threshold')
ax4.set_xlabel('Condition')
ax4.set_ylabel('Mean Score')
ax4.set_title('Mean Scores by Condition')
ax4.set_xticks(x)
ax4.set_xticklabels([labels[c] for c in condition_names])
ax4.legend()
ax4.set_ylim(0, 10)
ax4.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars1, cbt_means):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
            f'{val:.1f}', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, persona_means):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
            f'{val:.1f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()

# Save figure
fig_path = OUTPUT_DIR / "images" / "three_condition_comparison.png"
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFigure saved to {fig_path}")

## 9. Save Results

In [ ]:
# Save comprehensive results
results_data = {
    "metadata": {
        "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        "model": MODEL,
        "max_turns_processed": MAX_TURNS_TO_PROCESS,
        "experiment": "Three-Condition Comparison: Baseline vs Mem0 vs DC-RS"
    },
    "summary_comparison": {},
    "detailed_results": {}
}

for condition_name, results in all_results.items():
    results_data["summary_comparison"][condition_name] = results["summary"]
    results_data["detailed_results"][condition_name] = {
        "cbt_evaluations": results["cbt_evaluations"],
        "persona_evaluations": results["persona_evaluations"],
        "generated_responses": results["generated_responses"][:10],  # Sample
    }
    if "cheatsheet_snapshots" in results:
        results_data["detailed_results"][condition_name]["cheatsheet_snapshots"] = results["cheatsheet_snapshots"]
    if "final_cheatsheet" in results:
        results_data["detailed_results"][condition_name]["final_cheatsheet"] = results["final_cheatsheet"]
    if "memory_snapshots" in results:
        results_data["detailed_results"][condition_name]["memory_snapshots"] = results["memory_snapshots"]

# Calculate improvements
if "baseline" in all_results and "dcrs" in all_results:
    results_data["improvements"] = {
        "dcrs_vs_baseline": {
            "cbt": all_results["dcrs"]["summary"]["cbt_mean"] - all_results["baseline"]["summary"]["cbt_mean"],
            "persona": all_results["dcrs"]["summary"]["persona_mean"] - all_results["baseline"]["summary"]["persona_mean"]
        }
    }
    if "mem0" in all_results:
        results_data["improvements"]["dcrs_vs_mem0"] = {
            "cbt": all_results["dcrs"]["summary"]["cbt_mean"] - all_results["mem0"]["summary"]["cbt_mean"],
            "persona": all_results["dcrs"]["summary"]["persona_mean"] - all_results["mem0"]["summary"]["persona_mean"]
        }

results_path = OUTPUT_DIR / "results" / "three_condition_comparison.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results_data, f, indent=2, ensure_ascii=False)

print(f"Results saved to {results_path}")

## 10. Conclusions

### Key Findings

| Metric | Baseline | Mem0 | DC-RS | DC-RS vs Baseline | DC-RS vs Mem0 |
|--------|----------|------|-------|-------------------|---------------|
| CBT Mean | X.X | X.X | X.X | +X.X | +X.X |
| Persona Mean | X.X | X.X | X.X | +X.X | +X.X |

### Paper Claims

1. **"Continual test-time learning (DC-RS) outperforms static memory (mem0)"**
   - DC-RS extracts transferable STRATEGIES, not raw memories
   - Clinical filtering prevents distortion accumulation

2. **"Curated knowledge beats raw accumulation for therapeutic contexts"**
   - Cheatsheet stores patterns, not patient statements as facts
   - Maintains professional boundaries through curated templates

3. **"Test-time adaptation improves long-context therapeutic alignment"**
   - Strategies evolve during conversation
   - Effective interventions are reinforced, distortions are filtered

In [ ]:
print("="*60)
print("EXPERIMENT COMPLETE")
print("="*60)
print(f"\nThree-Condition Comparison: Baseline vs Mem0 vs DC-RS")
print(f"Model: {MODEL}")
print(f"Turns processed: {MAX_TURNS_TO_PROCESS}")
print(f"\nResults saved to: {OUTPUT_DIR}")
print(f"\nKey insight: DC-RS uses CONTINUAL TEST-TIME LEARNING")
print(f"to extract curated strategies rather than accumulating raw memories.")